# 09E — Application data mart

**Goal.** Export the compact, validated, versioned datasets that the **Combine Intelligence Lab** reads. The application never queries the warehouse: normal navigation touches only this directory.

```
warehouse/gold/advanced_movement/intelligence_v1/app_mart/v1/
    prospect_profiles.parquet        combine_attempt_curves.parquet     position_curve_bands.parquet
    prospect_metrics.parquet         feature_evidence.parquet
    stopwatch_twins.parquet          twin_differences.parquet           twin_summary.parquet
    nfl_movement_demands.parquet     nfl_demand_profiles.parquet        nfl_demand_distributions.parquet
    nfl_demand_context.parquet       nfl_context_availability.parquet   nfl_demand_registry.parquet
    nfl_demand_examples.parquet      (optional: 09A with BUILD_EXAMPLES = True)
    movement_similarity.parquet      movement_similarity_matrix.parquet movement_matching_summary.parquet
    shared_feature_dictionary.parquet
    scouting_cases.parquet           case_views.parquet                 case_nfl_observed.parquet
    interpretation_registry.parquet
    scouting_evidence_registry.json  mart_manifest.json
```

Nothing is computed here beyond joins and renames: every number was produced by 07D, 08B and 09A–09D. Provenance (source table of each file, run mode, fingerprints) is in the manifest.

**Validation** (`milestone09_artifacts.validate_mart`): declared version · every table present with its required columns · row counts and fingerprints equal to the manifest · every displayed athlete exists in the profile table · every case is a ranked twin pair with three views · no case selected on an NFL outcome · no grade or ranking column · similarity labelled descriptive · abstentions carry no interval · football value never claimed.

**Prerequisites (same `RUN_MODE`).** 01, 02, 07D, 09A, 09B, 09C, 09D. Optional: 08B.4 and 08B.6 (Research Evidence page).

In [1]:
RUN_MODE = 'full'             # 'smoke' | 'full' — a smoke mart is written under _sample and labelled as a wiring check by the application
SEED = 20261011
FORCE_REBUILD = False         # interface parity: the mart is rewritten on each run (seconds)
SAVE_TABLES = True            # False = build and validate in memory only (nothing written)
SAVE_FIGURES = True
NUM_WORKERS = 0
EXPERIMENT_VERSION = '09E.1'

In [2]:
from nflbdb.data.config import Phase01Config, configure_runtime

cfg = Phase01Config.load()
configure_runtime(cfg)

import json

import polars as pl
from IPython.display import Markdown, display

from nflbdb.research.advanced_movement import checkpoint as ck
from nflbdb.research.advanced_movement import config as amc
from nflbdb.research.advanced_movement import evidence_package as ep
from nflbdb.research.advanced_movement import intelligence_validation as iv
from nflbdb.research.advanced_movement import milestone09_artifacts as m9
from nflbdb.research.advanced_movement import movement_demands as md
from nflbdb.research.advanced_movement import movement_information as mi
from nflbdb.research.advanced_movement import movement_matching as mm
from nflbdb.research.advanced_movement import sequences as seq
from nflbdb.research.advanced_movement import stopwatch_twins as stw
from nflbdb.research.advanced_movement.artifacts import IntelligenceStore
from nflbdb.research.advanced_movement.scoring import scorecards as sc

controls = amc.NotebookControls(run_mode=RUN_MODE, force_rebuild=FORCE_REBUILD, save_figures=SAVE_FIGURES, save_tables=SAVE_TABLES,
                                seed=SEED, num_workers=NUM_WORKERS).validate()
store = IntelligenceStore(cfg, controls)
up, SC, FB = store.upstream, store.scoring, store.football
NB = {'01': '01_sequence_eda_and_alignment.ipynb', '02': '02_functional_pca_baseline.ipynb', '07D': '07d_athletic_fingerprint_scorecard.ipynb',
      '09A': '09a_nfl_movement_demand_atlas.ipynb', '09B': '09b_cross_domain_movement_matching.ipynb', '09C': '09c_stopwatch_twins_discovery.ipynb',
      '09D': '09d_football_interpretation_and_case_studies.ipynb', '09E': '09e_application_data_mart.ipynb'}

profile = SC.require(sc.PROFILE, NB['07D'])
frames = up.require(seq.SEQ_FRAMES, NB['01'])
windows = up.require(ck.WINDOWS, NB['02'])
A = {c.name: store.require(c, NB['09A']) for c in (md.REGISTRY, md.FREQUENCY, md.PROFILE, md.CONTEXT, md.AVAILABILITY, md.DISTRIBUTION)}
B = {c.name: store.require(c, NB['09B']) for c in (mm.PROSPECT, mm.EVIDENCE, mm.DICTIONARY, mm.SIMILARITY, mm.MATRIX, mm.DIM_SUMMARY)}
C = {c.name: store.require(c, NB['09C']) for c in (stw.TWINS, stw.DIFFS, stw.SUMMARY)}
D = {c.name: store.require(c, NB['09D']) for c in (stw.CASES, m9.VIEWS, m9.INTERPRETATION, m9.CASE_NFL)}
examples = store.read(md.EXAMPLES) if store.exists(md.EXAMPLES) else None
registry08 = FB.read(mi.REGISTRY) if FB.exists(mi.REGISTRY) else None
claims = FB.read(ep.EVIDENCE) if FB.exists(ep.EVIDENCE) else None
for what, df in (('08B.4 scouting evidence registry', registry08), ('08B.6 evidence claims', claims), ('09A example episodes', examples)):
    if df is None:
        print(f'OPTIONAL INPUT ABSENT — {what}: the application shows that section as not available')
label = 'SMOKE — wiring check, numbers are not results' if controls.smoke else 'FULL'
display(Markdown(f'**Run mode: `{controls.run_mode}`** ({label}) · mart version `{m9.MART_VERSION}` · target `{cfg.paths.rel(m9.mart_path(store))}`'))

**Run mode: `full`** (FULL) · mart version `v1` · target `warehouse/gold/advanced_movement/intelligence_v1/app_mart/v1`

## 1 · Synthetic checks of the mart contract

A complete mart passes; another declared version, a changed file, a missing table, a grade column, an unknown athlete, an unbacked case, an interval on an abstention and a football-value claim are each caught; an incomplete mart is not written.

In [3]:
validation = iv.run_all(iv.CHECKS_MART)
with pl.Config(fmt_str_lengths=260, tbl_rows=-1):
    display(validation)
iv.assert_all_passed(validation)

check,status,detail
str,str,str
"""application_mart_contract""","""PASS""","""the mart declares its version, matches its manifest, holds no grade column and refuses to be written incomplete"""
"""mart_referential_integrity""","""PASS""","""every displayed athlete exists in the profile table; cases, abstentions and football-value wording are enforced"""


## 2 · Assemble the application tables

In [4]:
twins = C[stw.TWINS.name]
TABLES = {
    'prospect_profiles': m9.profiles_table(profile, B[mm.PROSPECT.name], twins),
    'prospect_metrics': B[mm.PROSPECT.name],
    'combine_attempt_curves': m9.curves_table(frames, windows),
    'position_curve_bands': m9.bands_table(frames, windows, min_curves=3 if controls.smoke else 10),
    'feature_evidence': B[mm.EVIDENCE.name],
    'stopwatch_twins': twins,
    'twin_differences': C[stw.DIFFS.name],
    'twin_summary': C[stw.SUMMARY.name],
    'nfl_movement_demands': m9.demands_table(A[md.FREQUENCY.name], A[md.REGISTRY.name]),
    'nfl_demand_profiles': A[md.PROFILE.name],
    'nfl_demand_distributions': A[md.DISTRIBUTION.name],
    'nfl_demand_context': A[md.CONTEXT.name],
    'nfl_context_availability': A[md.AVAILABILITY.name],
    'nfl_demand_registry': A[md.REGISTRY.name],
    'movement_similarity': B[mm.SIMILARITY.name],
    'movement_similarity_matrix': B[mm.MATRIX.name],
    'movement_matching_summary': B[mm.DIM_SUMMARY.name],
    'shared_feature_dictionary': B[mm.DICTIONARY.name],
    'scouting_cases': D[stw.CASES.name],
    'case_views': D[m9.VIEWS.name],
    'case_nfl_observed': D[m9.CASE_NFL.name],
    'interpretation_registry': D[m9.INTERPRETATION.name],
}
if examples is not None:
    TABLES['nfl_demand_examples'] = examples
SOURCES = {'prospect_profiles': '07D combine_athletic_scorecard_profiles + 09B / 09C counts', 'prospect_metrics': '09B combine_curve_prospect_features',
           'combine_attempt_curves': '01 sequence_frames + 02 sequence_window_flags', 'position_curve_bands': '01 sequence_frames (07D curve_band)',
           'feature_evidence': '09B combine_curve_feature_evidence (+ 08B.4 registry statuses)', 'stopwatch_twins': '09C', 'twin_differences': '09C', 'twin_summary': '09C',
           'nfl_*': '09A (Milestone 05A eligible player-plays, silver.game_tracking)', 'movement_*': '09B', 'scouting_cases': '09D', 'case_views': '09D',
           'scouting_evidence_registry.json': '08B.4 scouting_evidence_registry + 08B.6 final_evidence_registry + 09D interpretation registry'}
with pl.Config(tbl_rows=-1):
    display(pl.DataFrame([{'table': k, 'rows': v.height, 'columns': v.width} for k, v in TABLES.items()]))

table,rows,columns
str,i64,i64
"""prospect_profiles""",510,31
"""prospect_metrics""",4080,13
"""combine_attempt_curves""",41500,9
"""position_curve_bands""",405,6
"""feature_evidence""",8,16
"""stopwatch_twins""",3266,27
"""twin_differences""",26128,17
"""twin_summary""",48,8
"""nfl_movement_demands""",54,15


## 3 · Write and validate

In [5]:
EVIDENCE_DOC = m9.evidence_document(registry08.to_dicts() if registry08 is not None else [], claims.to_dicts() if claims is not None else [],
                                    D[m9.INTERPRETATION.name].to_dicts(), B[mm.EVIDENCE.name].to_dicts(), SOURCES)
PROV = {'notebook': NB['09E'], 'experiment_version': EXPERIMENT_VERSION, 'run_mode': controls.run_mode, 'seed': controls.seed, 'sources': SOURCES}
if controls.save_tables:
    manifest = m9.write_mart(store, TABLES, EVIDENCE_DOC, PROV)
    report = m9.validate_mart(m9.mart_path(store))
    with pl.Config(fmt_str_lengths=200, tbl_rows=-1):
        display(report)
    print(json.dumps(manifest['overview'], indent=1))
    if registry08 is None:
        print('NOTE: evidence_document_present fails without the 08B.4 registry — run 08B.4 in this RUN_MODE, then rerun this notebook')
    m9.assert_valid(report)
    print('application mart written and validated:', cfg.paths.rel(m9.mart_path(store)))
else:
    print(json.dumps(m9.overview(TABLES), indent=1))
    print('SAVE_TABLES is False: the mart was assembled in memory and NOT written; the application cannot read it')

check,passed,detail
str,bool,str
"""declared_version""",true,"""manifest 'v1', expected 'v1'"""
"""tables_present""",true,"""all present"""
"""required_columns""",true,"""all present"""
"""rows_match_manifest""",true,"""row counts equal the manifest"""
"""fingerprints_match_manifest""",true,"""files unchanged since the manifest was written"""
"""profile_key_unique""",true,"""one profile row per athlete"""
"""displayed_athletes_exist_in_source""",true,"""every athlete of every table is in the profile table"""
"""cases_backed_by_twin_pairs""",true,"""23 case(s), each a ranked twin pair"""
"""no_case_selected_on_nfl_outcome""",true,"""no NFL outcome value entered a selection"""


{
 "n_prospects": 510,
 "n_tracked_prospects": 418,
 "positions": [
  "DB",
  "DL",
  "OL",
  "TE",
  "WR"
 ],
 "n_tracked_attempts": 793,
 "n_nfl_episodes": 416507,
 "n_demands_measurable": 9,
 "n_twin_pairs": 3266,
 "n_twin_pairs_distinguishable": 2097,
 "n_measurements": 8,
 "n_cases": 23,
 "n_case_types": 10,
 "n_reports": 510
}
application mart written and validated: warehouse/gold/advanced_movement/intelligence_v1/app_mart/v1


## 4 · Launching the application (you run this, in a terminal of your choice)

```
pip install -r apps/combine_intelligence_lab/requirements.txt      # once, into the environment you will launch from
streamlit run apps/combine_intelligence_lab/app.py
```

The application looks for the mart at `warehouse/gold/advanced_movement/intelligence_v1/app_mart/v1` under the project root. To point it elsewhere (for example at the smoke mart) set the environment variable `CIL_MART_DIR` to the mart directory before launching. Details: `apps/combine_intelligence_lab/README.md`.

**Send back:** the checks table, the table of rows per mart file, the validation table and the overview.

**Next:** `09f_intelligence_lab_demonstration.ipynb` (the whole product workflow in a notebook, from this mart only).